# Lesson 02 Lab — Tensor Core Constraints for Low-Precision GEMM

**Puzzle:** A low-precision dtype is available, so will every matrix multiplication automatically become a fast Tensor Core operation?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A GEMM consumes `A[M,K]` and `B[K,N]`. Dtype, strides, transposition, leading dimensions, and the three logical sizes travel together into dispatch; the word *BF16* by itself is not a kernel description.

### Core mechanism

A useful first model is `FLOPs ≈ 2MKN` and `arithmetic intensity = FLOPs / bytes moved`. Large aligned tiles can amortize loads and feed matrix-multiply hardware; awkward dimensions create edge tiles, padding, or a different implementation. Tensor Core eligibility is therefore a conjunction of hardware, dtype, shape, layout, and library support.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "02-tensor-core-constraints"
device = require_cuda()
torch.manual_seed(2026 + 2)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Padding may improve tile utilization but adds work and memory. Small GEMMs may be launch- or memory-dominated, so a lower-precision peak-FLOP number may never become the bottleneck that the application sees.

### What this code tests

The lab changes dtype and one alignment condition while keeping the GPU and timing method fixed; the output is shape evidence, not a native-kernel assertion.

**Experiment:** Time FP32 and BF16 matrix multiplications with aligned and deliberately awkward dimensions on the same GPU.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
shapes = {"aligned": (2048, 2048, 2048), "awkward": (2048, 2055, 2048)}
timings = {}
for name, (m, k, n) in shapes.items():
    timings[name] = {}
    for dtype in (torch.float32, torch.bfloat16):
        a = torch.randn(m, k, device=device, dtype=dtype)
        b = torch.randn(k, n, device=device, dtype=dtype)
        timings[name][str(dtype).split(".")[-1]] = cuda_benchmark(lambda: a @ b, warmup=4, repeats=12)
result = base_result(2, "pytorch-gpu")
result.update({"shapes_mkn": shapes, "timings": timings,
               "conclusion": "Observed shape- and dtype-dependent GEMM timing; native Tensor Core identity requires a lower-level profiler."})


## 3. Inspect the evidence

Compare medians by dtype and shape. The lab does not infer Tensor Core use from speed alone; it records a PyTorch GPU timing baseline for later profiler work.

### Acceptance and rollback gate

Keep the exact `M,N,K`, strides, dtype, warm-up, and repeated timing. Use an operator trace to show dispatch and Nsight Compute/System metrics before naming a native Tensor Core kernel.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Observed shape- and dtype-dependent GEMM timing; native Tensor Core identity requires a lower-level profiler.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:07+00:00",
  "lesson": 2,
  "schema_version": 1,
  "shapes_mkn": {
    "aligned": [
      2048,
      2048,
      2048
    ],
    "awkward": [
      2048,
      2055,
      2048
    ]
  },
  "timings": {
    "aligned": {
      "bfloat16": {
        "median_ms": 0.087632,
        "p90_ms": 0.088864,
        "repeats": 12,
        "samples_ms": [
          0.0936,
          0.08896,
          0.08768,
          0.087584,
          0.086816,
          0.087904,
          0.086752,
          0.086528,
          0.088864,
          0.087104,
          0.087872,
          0.087456
        ],
 

## 4. Explain the result

Low precision creates an opportunity, not a guarantee. Preserve exact shapes and profiler evidence when deciding whether a Tensor Core path was reached.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).